# The week's runs: Amendments 27, 28 and 29 (all exploratory or audits)

All sessions: **CPU + High-RAM**, notebook **saved in your Drive** (File → Save a copy in Drive), the same commit SHA in cell 1.

| Session | ROLE | What it does | Cells |
|---|---|---|---|
| 1 | `A0` | re-executes half of the Amendment 26 runs (1,200) | 1–7, then 8 |
| 2 | `A1` | the other half (1,200) | 1–7, then 8 |
| 3 | `C` | squash check (1,600 runs) + its re-execution + analysis; the 6 full-data runs without clipping; learning-rate check for the classical arms | 1–7, then 9 → 14 |
| 4 | `D1` | learning-rate check: re-upload, range-1, alternative ansatz | 1–7, then 15 |
| 5 | `D2` | learning-rate check: both readouts, VQC at d = 8 | 1–7, then 15 |
| 6, 7 (if time) | `E1`, `E2` | re-execute the learning-rate check | 1–7, then 16 |
| any, at the end | same as that session | learning-rate analysis; verdict counts | 17, 18 |

If Colab will not give you five sessions at once, start D1 and D2 when A0 and A1 finish (change ROLE, run 1–7 again).

**Rules learned in earlier sessions:** every cell must end with `OK`; if not, stop and send the output. Results go to Drive, so a disconnect loses at most the run in progress: run cells 1–7 again, then the cell you were on; finished runs are skipped. No cell prints a result while runs are still going.

In [ ]:
# 1. SETTINGS. Edit these two lines only.
EXPECTED_SHA = "PASTE_THE_FULL_SHA_OF_THE_AMENDMENT_27_28_COMMIT"
ROLE = "A0"            # "A0", "A1", "C", "D1", "D2", "E1" or "E2"

BACKUP = "/content/drive/MyDrive/QML-backup-run"
EXP31  = BACKUP + "/exp31"                 # Amendment 26 results (read only)
CPU17  = BACKUP + "/cpu17_out"             # Amendment 25 results (read only)
OUT    = BACKUP + "/cpu_reruns"            # everything this notebook writes
REPO, ART, ENV = "/content/repo", "/content/art", "/content/env"
PY = ENV + "/bin/python"
SHA26, SHA25 = "58852d84", "e07459a5"      # commits that recorded 31_entangling_range and 17_fulldata
DATASETS = ["breastmnist", "pneumoniamnist", "bloodmnist", "pathmnist"]
REGIMES  = ["5", "10", "20", "50", "100"]
ARMS26   = ["quantum_vqc", "quantum_vqc_range1", "quantum_reupload"]
T28      = 2                               # threads per process for the squash runs and their re-execution
T34      = 2                               # threads per process for the learning-rate check and its re-execution
TUNE_SEEDS = [90001, 90002, 90003, 90004, 90005]
TUNE = {   # Amendment 29: (arm, rank, d) per role
    "C":  [("low_rank", 0, 4), ("low_rank", 2, 4), ("low_rank", 8, 4), ("fourier_rff_r2", None, 4), ("low_rank", 2, 8)],
    "D1": [("quantum_reupload", None, 4), ("quantum_vqc_range1", None, 4), ("quantum_basic", None, 4)],
    "D2": [("quantum_rich", None, 4), ("quantum_rich_padded", None, 4), ("quantum_vqc", None, 8)],
}
THREADS_OVERRIDE = None                    # set only if cell 8 asks you to

import os, subprocess, json, glob, time, shutil, re
assert ROLE in ("A0", "A1", "C", "D1", "D2", "E1", "E2"), ROLE
os.environ.update({"QML_ARTIFACT_ROOT": ART, "MPLBACKEND": "Agg", "CUDA_VISIBLE_DEVICES": ""})

def sh(cmd, log=None, check=True, env=None):
    """Run a bash command from the repo, print its output, append it to a log on Drive."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log:
        os.makedirs(OUT + "/logs", exist_ok=True)
        with open(f"{OUT}/logs/{log}", "a") as f:
            f.write(f"\n$ {cmd}\n" + "".join(lines) + f"exit code {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"FAILED (exit {rc}): {cmd}")
    return "".join(lines)

def launch(cmds, threads, tag):
    """Start commands side by side in the background, each with `threads` threads. Logs on Drive."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    procs = {}
    for name, cmd in cmds.items():
        log = f"{OUT}/logs/{tag}_{name}.txt"
        procs[name] = subprocess.Popen(["bash", "-lc", f"{cmd} >> {log} 2>&1"], cwd=REPO,
                                       env=env, start_new_session=True)
        print(f"started {name} (pid {procs[name].pid}, {threads} threads), log: {log}")
    return procs

def wait(procs, every=900, show=None):
    """Wait for background processes; print progress every 15 minutes."""
    while any(p.poll() is None for p in procs.values()):
        if show: show()
        time.sleep(every)
    if show: show()
    codes = {n: p.returncode for n, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"

def verdicts(target):
    """Count replay verdicts in a target namespace on Drive."""
    from collections import Counter
    c = Counter()
    for fn in glob.glob(f"{OUT}/shards/{target}/*.json"):
        try:
            v = json.load(open(fn)).get("replay_check", {}).get("verdict", "?")
        except Exception:
            v = "unreadable"
        c[v.split(" (")[0]] += 1
    return dict(c)
def squash_cmds(seeds):
    """Amendment 28: the matched head with and without the squash, side by side."""
    base = (f"{PY} -u src/01_frozen_backbone_ablation.py --arms matched_param_fullrank --dims 4 "
            f"--datasets {' '.join(DATASETS)} --regimes {' '.join(REGIMES)} "
            f"--seeds {' '.join(map(str, seeds))} --lr-head 1e-2 --experiment 32_squash")
    return {"with_squash": base + " --with-tanh", "without_squash": base + " --no-tanh"}
def tune_cmds(specs, seeds):
    """Amendment 29: the unchanged tuning procedure, one process per (arm, rank, d)."""
    out = {}
    for arm, rank, d in specs:
        name = f"{arm}{'' if rank is None else f'_r{rank}'}_d{d}"
        out[name] = (f"{PY} -u src/09_lr_selection.py --experiment 34_lr_inherited --arms {arm} --dim {d} "
                     f"--seeds {' '.join(map(str, seeds))}" + ("" if rank is None else f" --head-rank {rank}"))
    return out

def count34():
    return len(glob.glob(f"{OUT}/shards/34_lr_inherited/*.json"))
print("ROLE", ROLE, "OK")

In [ ]:
# 2. MOUNT DRIVE and check the inputs exist.
from google.colab import drive
drive.mount("/content/drive")
fc = sorted(glob.glob(BACKUP + "/feature_cache_*.tgz"))
assert fc, "feature_cache_*.tgz not found"
assert len(glob.glob(EXP31 + "/shards/31_entangling_range/*.json")) == 2400, "Amendment 26 shards missing on Drive"
if ROLE == "C":
    rb = sorted(glob.glob(BACKUP + "/replay_backup_*.tgz"))
    assert rb, "replay_backup_*.tgz not found (needed for the full-data runs)"
    assert len(glob.glob(CPU17 + "/shards/C_17_fulldata/*.json")) == 80, "cpu17_out C_17_fulldata incomplete"
os.makedirs(OUT + "/logs", exist_ok=True)
print("feature cache:", fc[-1])
print("OK")

In [ ]:
# 3. CLONE THE REPO at the pushed commit; check it is clean and carries Amendments 27 and 28.
if not os.path.isdir(REPO):
    sh(f"git clone -b feature/journal-expansion https://github.com/anishfelixm/quantum-classical-expressivity {REPO}")
sh(f"git fetch origin && git checkout --detach {EXPECTED_SHA}")
assert sh("git rev-parse HEAD").strip() == EXPECTED_SHA
assert sh("git status --porcelain").strip() == "", "the clone has local changes"
plan = open(REPO + "/docs/analysis_plan.md").read()
assert all(f"## Amendment {a}" in plan for a in (27, 28, 29)), "Amendments 27-29 missing at this commit"
for f in ["scripts/replay.py", "scripts/analyse_squash.py", "scripts/analyse_lr_inherited.py"]:
    assert os.path.exists(f"{REPO}/{f}"), f
assert "CU_" in open(REPO + "/scripts/replay.py").read(), "replay.py at this commit lacks the CU_ target"
assert "--head-rank" in open(REPO + "/src/09_lr_selection.py").read(), "09_lr_selection.py lacks --head-rank"
print("OK")

In [ ]:
# 4. PYTHON ENVIRONMENT: Python 3.10 with requirements.lock.txt (about 5 minutes).
if not os.path.exists(PY):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba")
    sh(f"/content/bin/micromamba create -y -q -p {ENV} -c conda-forge python=3.10 pip")
    sh(f"{PY} -m pip install -q -r requirements.lock.txt pytest --extra-index-url https://download.pytorch.org/whl/cu118")
sh("apt-get -qq install -y pigz > /dev/null", check=False)
sh(f"{PY} -c \"import sys, torch; from importlib.metadata import version as v; print('python', sys.version.split()[0], '| torch', torch.__version__, '| pennylane', v('pennylane'), '| numpy', v('numpy'), '| scikit-learn', v('scikit-learn'), '| medmnist', v('medmnist'), '| default threads', torch.get_num_threads())\"",
   log="environment.txt")
print("OK")

In [ ]:
# 5. DEVICE. CPU only; record this session (cores, processor, role).
info = sh(f"{PY} -c \"import sys; sys.path.insert(0,'src'); import config; print(config.DEVICE.type)\"").split()
assert info[-1] == "cpu", "Use the CPU runtime (Runtime -> Change runtime type -> CPU, High-RAM)."
cpu = sh("lscpu | grep 'Model name' | sed 's/.*: *//'", check=False).strip()
cores = os.cpu_count()
with open(OUT + "/logs/sessions.txt", "a") as f:
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')} role={ROLE} cpu='{cpu}' cores={cores} sha={EXPECTED_SHA[:8]}\n")
print(f"cpu='{cpu}'  cores={cores}")
print("OK")

In [ ]:
# 6. GATES: parity and circuit-wiring tests (the code that produced the runs is re-checked here).
sh(f"{PY} -m pytest -q tests/test_parity.py tests/test_entangling_ranges.py", log=f"gates_{ROLE}.txt")
print("OK")

In [ ]:
# 7. ARTIFACT FOLDER: links to Drive and the restored feature cache.
def link(sub, ns, target_dir):
    os.makedirs(f"{ART}/{sub}", exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    p = f"{ART}/{sub}/{ns}"
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(target_dir, p)

# read: the Amendment 26 records
for sub in ("shards", "predictions"):
    link(sub, "31_entangling_range", f"{EXP31}/{sub}/31_entangling_range")
# write: every target of this notebook, on Drive
for ns in ("C_31_entangling_range", "32_squash", "C_32_squash", "CU_17_fulldata",
           "34_lr_inherited", "C_34_lr_inherited"):
    for sub in ("shards", "predictions", "step_norms", "weights"):
        link(sub, ns, f"{OUT}/{sub}/{ns}")

# the 10 October feature cache (config.py creates an empty folder on import, so test for the link)
FC = f"{ART}/feature_cache"
if not os.path.islink(FC):
    if os.path.isdir(FC):
        shutil.rmtree(FC)
    pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    if not pts:
        os.makedirs("/content/fc_extract", exist_ok=True)
        sh(f"tar -I pigz -xf '{fc[-1]}' -C /content/fc_extract")
        pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    os.symlink(os.path.dirname(pts[0]), FC)
assert os.path.realpath(FC).startswith("/content/fc_extract")
seeds_all = json.loads(sh(f"{PY} -c \"import sys, json; sys.path.insert(0,'src'); import config; "
                          f"print(json.dumps(config.CONFIRMATORY_SEEDS))\"").strip().splitlines()[-1])
missing = [(d, r, s) for d in DATASETS for r in REGIMES for s in seeds_all
           if not os.path.exists(f"{FC}/{d}__r{r}__s{s}.pt")]
assert not missing, f"{len(missing)} cache files missing; stop and send this output"

# Amendment 29: the tuning seeds' caches (n = 5, 20, 100). Missing ones are built in a SEPARATE
# process, never inside a training run (a run that builds its own cache draws its start differently).
if ROLE in ("C", "D1", "D2", "E1", "E2"):
    need = [(d, r, s) for d in DATASETS for r in ("5", "20", "100") for s in TUNE_SEEDS]
    miss = [t for t in need if not os.path.exists(f"{FC}/{t[0]}__r{t[1]}__s{t[2]}.pt")]
    print(f"tuning-seed cache files present: {len(need) - len(miss)} of {len(need)}")
    # Any file not in the backup is built ONCE, by session C, and shared through Drive, so every
    # session (and any later re-execution) uses the same bytes.
    TC = f"{OUT}/tune_cache"
    os.makedirs(TC, exist_ok=True)
    for t in miss:
        f = f"{t[0]}__r{t[1]}__s{t[2]}.pt"
        if os.path.exists(f"{TC}/{f}"):
            shutil.copy(f"{TC}/{f}", f"{FC}/{f}")
    miss = [t for t in need if not os.path.exists(f"{FC}/{t[0]}__r{t[1]}__s{t[2]}.pt")]
    if miss and ROLE != "C":
        raise RuntimeError(f"{len(miss)} tuning cache files are missing. Run cells 1-7 of session C first "
                           "(it builds them once and saves them to Drive), then run this cell again.")
    if miss:
        json.dump(miss, open("/content/missing_tune.json", "w"))
        open("/content/warm_tune.py", "w").write(
            "import sys, json, importlib; sys.path.insert(0, 'src')\n"
            "m = importlib.import_module('01_frozen_backbone_ablation')\n"
            "for d, r, s in json.load(open('/content/missing_tune.json')):\n"
            "    m.get_cached_features(d, r, s); print('built', d, r, s, flush=True)\n")
        sh(f"{PY} /content/warm_tune.py", log="cache_prewarm_tune.txt")
        built = [f"{t[0]}__r{t[1]}__s{t[2]}.pt" for t in miss]
        miss = [t for t in need if not os.path.exists(f"{FC}/{t[0]}__r{t[1]}__s{t[2]}.pt")]
        assert not miss, f"{len(miss)} tuning cache files still missing"
        for f in built:
            shutil.copy(f"{FC}/{f}", f"{TC}/{f}")
        print(f"built {len(built)} tuning cache files and saved them to Drive for the other sessions")

if ROLE == "C":
    # the full-data records (Amendment 25 inputs): 17_fulldata from the replay backup, C_17_fulldata from
    # cpu17_out. The backup has no data_cache; MedMNIST downloads it when needed (found 8 November).
    if len(glob.glob("/content/rb_extract/**/shards/17_fulldata/*.json", recursive=True)) < 80:
        os.makedirs("/content/rb_extract", exist_ok=True)
        sh(f"tar -I pigz -xf '{rb[-1]}' -C /content/rb_extract --wildcards "
           f"'*artifacts/shards/17_fulldata/*' '*artifacts/predictions/17_fulldata/*'")
    root = glob.glob("/content/rb_extract/**/artifacts", recursive=True)[0]
    for sub in ("shards", "predictions"):
        link(sub, "17_fulldata", f"{root}/{sub}/17_fulldata")
    for sub in ("shards", "predictions", "step_norms", "weights"):
        link(sub, "C_17_fulldata", f"{CPU17}/{sub}/C_17_fulldata")
    dc = f"{ART}/data_cache"
    if os.path.islink(dc):
        os.unlink(dc)
    os.makedirs(dc, exist_ok=True)
    n17 = len(glob.glob(f"{ART}/shards/17_fulldata/*.json"))
    p17 = len(glob.glob(f"{ART}/predictions/17_fulldata/*"))
    c17 = len(glob.glob(f"{ART}/shards/C_17_fulldata/*.json"))
    full = len(glob.glob(f"{FC}/*__rfull__s*.pt"))
    print(f"17_fulldata records {n17}, predictions {p17} | C_17_fulldata {c17} | full-data feature caches {full}")
    assert n17 == 80 and p17 >= 80 and c17 == 80, "full-data records incomplete: send this output"
    assert full >= 20, "full-data feature caches missing from the backup: send this output"
print("cache files present: 800 of 800")
print("OK")

In [ ]:
# 8. ROLE A0 / A1 - AMENDMENT 27, BLOCK A: re-execute this session's Amendment 26 runs on CPU.
#    Same seeds, three arms side by side, and the same threads per process as the original session.
assert ROLE in ("A0", "A1")
k = int(ROLE[1])
my_seeds = seeds_all[k::2]

# threads of the original session k: max(1, cores // 3), cores from exp31/logs/sessions.txt
lines = [l for l in open(EXP31 + "/logs/sessions.txt") if f"session {k}/" in l and f"sha={SHA26}" in l]
print("".join(lines))
rec_cores = sorted({int(re.search(r"cores=(\d+)", l).group(1)) for l in lines})
rec_cpus = sorted({re.search(r"cpu='([^']*)'", l).group(1) for l in lines})
if THREADS_OVERRIDE is None:
    assert len(rec_cores) == 1, f"session {k} has more than one core count recorded {rec_cores}: send this output"
THREADS = THREADS_OVERRIDE or max(1, rec_cores[0] // 3)
print(f"original session {k}: cores={rec_cores} -> {THREADS} threads per process; processor {rec_cpus}")
if cpu not in rec_cpus:
    print(f"NOTE: this processor ('{cpu}') differs from the original's. Continue; the verdicts will show it.")

base = f"{PY} -u scripts/replay.py --source 31_entangling_range --clip 20 --code-at {SHA26}"
# a quick check first (one small run per arm, single process); it also creates the code worktree
sh(f"OMP_NUM_THREADS={THREADS} MKL_NUM_THREADS={THREADS} {base} --quick --seeds {' '.join(map(str, my_seeds))}",
   log=f"replay31_quick_{ROLE}.txt")
print("quick check:", verdicts("C_31_entangling_range"))
assert all(v.startswith("IDENTICAL") for v in verdicts("C_31_entangling_range")), "quick check not identical: stop, send the output"

procs = launch({arm: f"{base} --where arm={arm} --seeds {' '.join(map(str, my_seeds))}" for arm in ARMS26},
               THREADS, f"replay31_{ROLE}")
wait(procs, show=lambda: print(time.strftime("%H:%M"), "C_31 verdicts so far (all sessions):",
                               verdicts("C_31_entangling_range"), flush=True))
print("this session's re-execution is done. OK")

In [ ]:
# 9. ROLE C - AMENDMENT 28: the first two seeds of the squash check, timed; then a smoke test.
#    Both variants side by side, 2 threads each (fixed, so the re-execution can repeat it).
assert ROLE == "C"
t0 = time.time()
wait(launch(squash_cmds(seeds_all[:2]), T28, "squash_first2"), every=60)
dt = time.time() - t0
n_done = len(glob.glob(f"{OUT}/shards/32_squash/*.json"))
print(f"first 2 seeds: {n_done} runs in {dt/60:.1f} min -> the other 38 seeds about {dt*19/3600:.1f} h "
      f"(scaled from 2 seeds that include PathMNIST)")
sh(f"{PY} -u scripts/analyse_squash.py --smoke --recompute --workers {max(1, cores-1)}", log="squash_smoke.txt")
print("OK")

In [ ]:
# 10. ROLE C - AMENDMENT 28: the other 38 seeds.
assert ROLE == "C"
procs = launch(squash_cmds(seeds_all[2:]), T28, "squash_rest")
wait(procs, show=lambda: print(time.strftime("%H:%M"), "32_squash shards:",
                               len(glob.glob(f"{OUT}/shards/32_squash/*.json")), "of 1600", flush=True))
print("OK")

In [ ]:
# 11. ROLE C - AMENDMENT 28: re-execute every squash run on CPU (target C_32_squash), same threads.
assert ROLE == "C"
recs = [json.load(open(f)) for f in glob.glob(f"{OUT}/shards/32_squash/*.json")]
shas = sorted({r["git_sha"][:8] for r in recs})
assert len(recs) == 1600 and len(shas) == 1, (len(recs), shas)
base = f"{PY} -u scripts/replay.py --source 32_squash --clip 20 --code-at {shas[0]}"
sh(f"OMP_NUM_THREADS={T28} MKL_NUM_THREADS={T28} {base} --quick", log="replay32_quick.txt")   # creates the worktree
procs = launch({f"notanh{v}": f"{base} --where notanh={v}" for v in (0, 1)}, T28, "replay32")
wait(procs, show=lambda: print(time.strftime("%H:%M"), "C_32 verdicts:", verdicts("C_32_squash"), flush=True))
print("C_32_squash verdicts:", verdicts("C_32_squash"))
print("OK")

In [ ]:
# 12. ROLE C - AMENDMENT 28: integrity checks, then the pre-registered exploratory analysis (B = 2000).
assert ROLE == "C"
recs = [json.load(open(f)) for f in glob.glob(f"{OUT}/shards/32_squash/*.json")]
assert len(recs) == 1600, len(recs)
assert len({r["git_sha"] for r in recs}) == 1, "more than one commit in 32_squash"
no_pred = [r["keys"] for r in recs if not os.path.exists(
    f"{OUT}/predictions/32_squash/" + os.path.basename(r.get("predictions_file", "none")))]
assert not no_pred, f"{len(no_pred)} runs without predictions, e.g. {no_pred[:2]}"
out = sh(f"{PY} -u scripts/analyse_squash.py --workers {max(1, cores-1)}", log="stats_squash.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/squash_check.json", f"{OUT}/squash_check.json")
print("OK")

In [ ]:
# 13. ROLE C - AMENDMENT 27, BLOCK B: the full-data runs in which clipping acted on CPU, re-executed
#     without clipping on CPU (target CU_17_fulldata), one at a time with default threads, as Amendment 25.
assert ROLE == "C"
import numpy as np
sel = []
for fn in sorted(glob.glob(f"{ART}/shards/C_17_fulldata/*.json")):
    r = json.load(open(fn))
    sfile = os.path.join(ART, (r.get("recorded") or {}).get("step_norms_file", ""))
    with np.load(sfile) as z:
        if (~np.isfinite(z["norms"]) | (z["norms"] > 20)).any():
            sel.append(r)
print(f"runs with a step above 20 in C_17_fulldata: {len(sel)} (predicted 6)")
for r in sel:
    k = r["keys"]; print("   ", k["arm"], k["dataset"], k["seed"])
names = [os.path.basename(f) for f in glob.glob(f"{ART}/shards/17_fulldata/*.json")
         if any(json.load(open(f))["keys"] == r["keys"] for r in sel)]
open("/content/cu17_list.txt", "w").write("\n".join(names) + "\n")
assert len(names) == len(sel)
sh(f"{PY} -u scripts/replay.py --source 17_fulldata --clip inf --code-at {SHA25} --from-list /content/cu17_list.txt",
   log="replay17_unclipped.txt")
print("CU_17_fulldata verdicts:", verdicts("CU_17_fulldata"))

# descriptive: AUC change per run, and the mean full-data VQC - matched over (dataset, seed), clipped vs unclipped
def auc(r): return r["metrics"]["auc"]
def by_key(folder):
    out = {}
    for f in glob.glob(f"{folder}/*.json"):
        r = json.load(open(f)); out[json.dumps(r["keys"], sort_keys=True)] = r
    return out
rec, unc = by_key(f"{ART}/shards/17_fulldata"), by_key(f"{OUT}/shards/CU_17_fulldata")
for kk, r in sorted(unc.items()):
    k = r["keys"]
    print(f"   {k['arm']:24s} {k['dataset']:12s} s{k['seed']}: AUC {auc(rec[kk]):.4f} -> {auc(r):.4f} ({auc(r) - auc(rec[kk]):+.4f})")
def pick(arm, ds, seed, unclipped):
    hits = [kk for kk, r in rec.items() if r["keys"]["arm"] == arm and r["keys"]["dataset"] == ds
            and r["keys"]["seed"] == seed]
    assert len(hits) == 1, (arm, ds, seed, len(hits))
    kk = hits[0]
    return auc(unc[kk]) if unclipped and kk in unc else auc(rec[kk])
pairs = sorted({(r["keys"]["dataset"], r["keys"]["seed"]) for r in rec.values()})
for lab, u in (("clipped (as recorded)", False), ("clipping removed", True)):
    d = [pick("quantum_vqc", ds, s, u) - pick("matched_param_fullrank", ds, s, u) for ds, s in pairs]
    print(f"   full-data VQC - matched, mean over {len(d)} (dataset, seed) pairs, {lab}: {np.mean(d):+.4f}")
print("OK")

In [ ]:
# 14. ROLE C - AMENDMENT 29: learning-rate check for the classical arms (fast). First seed timed, then the rest.
assert ROLE == "C"
t0 = time.time()
wait(launch(tune_cmds(TUNE["C"], TUNE_SEEDS[:1]), T34, "tune_C_first"), every=120)
print(f"first tuning seed: {(time.time()-t0)/60:.1f} min -> the other 4 seeds about {(time.time()-t0)*4/3600:.1f} h")
wait(launch(tune_cmds(TUNE["C"], TUNE_SEEDS[1:]), T34, "tune_C_rest"),
     show=lambda: print(time.strftime("%H:%M"), "34_lr_inherited runs (all sessions):", count34(), "of 3960", flush=True))
print("OK")

In [ ]:
# 15. ROLE D1 / D2 - AMENDMENT 29: learning-rate check for the circuit arms. First seed timed, then the rest.
assert ROLE in ("D1", "D2")
t0 = time.time()
wait(launch(tune_cmds(TUNE[ROLE], TUNE_SEEDS[:1]), T34, f"tune_{ROLE}_first"),
     show=lambda: print(time.strftime("%H:%M"), "runs so far:", count34(), flush=True))
dt = time.time() - t0
print(f"first tuning seed: {dt/60:.1f} min -> the other 4 seeds about {dt*4/3600:.1f} h (scaled from a seed that includes PathMNIST)")
wait(launch(tune_cmds(TUNE[ROLE], TUNE_SEEDS[1:]), T34, f"tune_{ROLE}_rest"),
     show=lambda: print(time.strftime("%H:%M"), "34_lr_inherited runs (all sessions):", count34(), "of 3960", flush=True))
print("OK")

In [ ]:
# 16. ROLE E1 / E2 (only if time allows) - AMENDMENT 29: re-execute the learning-rate check on CPU (C_34_lr_inherited).
assert ROLE in ("E1", "E2")
recs = [json.load(open(f)) for f in glob.glob(f"{OUT}/shards/34_lr_inherited/*.json")]
shas = sorted({r["git_sha"][:8] for r in recs})
assert len(recs) == 3960 and len(shas) == 1, (len(recs), shas)
base = f"{PY} -u scripts/replay.py --source 34_lr_inherited --clip 20 --code-at {shas[0]}"
groups = {"E1": {"quantum_reupload": "arm=quantum_reupload", "quantum_vqc_range1": "arm=quantum_vqc_range1",
                 "quantum_basic": "arm=quantum_basic", "low_rank": "arm=low_rank", "fourier_rff_r2": "arm=fourier_rff_r2"},
          "E2": {"quantum_rich": "arm=quantum_rich", "quantum_rich_padded": "arm=quantum_rich_padded",
                 "quantum_vqc_d8": "arm=quantum_vqc"}}[ROLE]
sh(f"OMP_NUM_THREADS={T34} MKL_NUM_THREADS={T34} {base} --quick --where {list(groups.values())[0]}",
   log=f"replay34_quick_{ROLE}.txt")                                    # creates the code worktree first
procs = launch({n: f"{base} --where {w}" for n, w in groups.items()}, T34, f"replay34_{ROLE}")
wait(procs, show=lambda: print(time.strftime("%H:%M"), "C_34 verdicts:", verdicts("C_34_lr_inherited"), flush=True))
print("OK")

In [ ]:
# 17. ANY SESSION, after C (cell 14), D1 and D2 have all finished - AMENDMENT 29 analysis.
sh(f"{PY} -u scripts/analyse_lr_inherited.py --partial")
out = sh(f"{PY} -u scripts/analyse_lr_inherited.py", log="stats_lr_inherited.txt")
shutil.copy(f"{ART}/lr_inherited.json", f"{OUT}/lr_inherited.json")
print("OK")

In [ ]:
# 18. ANY SESSION, AT THE END: verdict counts of every re-execution block.
for t in ("C_31_entangling_range", "C_32_squash", "CU_17_fulldata", "C_34_lr_inherited"):
    print(f"{t:24s} {verdicts(t)}")
print("Expected: C_31 IDENTICAL 2400; C_32 IDENTICAL 1600; CU_17 PREFIX IDENTICAL 6; C_34 IDENTICAL 3960 (if run).")
print("OK")